In [1]:
!pip install langchain openai faiss-cpu pypdf pymupdf streamlit tiktoken

In [2]:
!pip install -U langchain-community

In [3]:
import fitz  # PyMuPDF

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text


In [ ]:
text = extract_text_from_pdf("example_2009.pdf")
print(text[:1000])  # Check first 1000 chars


arXiv:0902.1308v1  [astro-ph.GA]  8 Feb 2009
A New Supernova Remnant Candidate and an
Associated Outﬂow in the Sagittarius C Region
Takeshi Go Tsuru,1 Masayoshi Nobukawa,1 Hiroshi Nakajima,2 Hironori Matsumoto,1
Katsuji Koyama,1 Shigeo Yamauchi,3
1Department of Physics, Graduate School of Science, Kyoto University, Sakyo-ku, Kyoto, 606-8502
tsuru@cr.scphys.kyoto-u.ac.jp
2Department of Earth and Space Science, Graduate School of Science, Osaka University, Toyonaka,
Osaka, 560-0043
3Faculty of Humanities and Social Sciences, Iwate University, 3-18-34 Ueda, Morioka, Iwate
020-8550
(Received 2008 August 5; accepted 2008 September 15)
Abstract
We present the Suzaku results on a new candidate of a supernova remnant
(SNR) in the Sagittarius C region. We detected diﬀuse X-rays of an elliptical shape
(G 359.41−0.12) and chimney-like structure (the Chimney), both of which are ﬁtted
with a thin thermal model of kBT∼1 keV temperature. The absorption columns are
same between these two structures, i

## Step2: text splitting

In [5]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
docs = splitter.create_documents([text])


## Step3: embedding & vector store

In [11]:
!source ~/.bash_profile

In [14]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large")


In [15]:
db = FAISS.from_documents(docs, embeddings)


## Step4: Build the RetrievalQA Chain 

In [16]:
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI

retriever = db.as_retriever(search_type="similarity", search_kwargs={"k":4})

qa = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model="gpt-4o"),  # or gpt-4
    chain_type="stuff",
    retriever=retriever
)


In [18]:
### Test query:

query = "What are the main findings about the Chimney?"
answer = qa.run(query)
print(answer)


The main findings about the Chimney are:

1. The Chimney is a real structure that is physically connected to G 359.41−0.12. This connection is evidenced by a one-dimensional surface brightness profile at 2.45 keV, showing that the Chimney is emanating from G 359.41−0.12.

2. The Chimney appears to be an outflow plasma from a supernova remnant (SNR). Its morphology is unusual, characterized by a chimney-like thermal outflow with the dimensions approximately 8 pc × 8 pc × 34 pc.

3. Molecular clouds, specifically MC 3 and MC 4, block the plasma from expanding, except in the northern direction, which may lead to the formation of the outflow towards the north.

4. The Chimney and G 359.41−0.12 share the same absorption columns, indicating they are located at the same distance and in the same line of sight.

5. The combined thermal energy of the Chimney and G 359.41−0.12 is estimated to be 1.4×10^50 ergs.

6. The Chimney is not related to the non-thermal radio filament G 359.45−0.06, as the

## Step5: Build a streamlit chatbot

In [19]:
import streamlit as st

st.title("Astronomy Paper QA Chatbot")

query = st.text_input("Ask a question about the paper:")

if query:
    answer = qa.run(query)
    st.write(answer)


2025-07-03 12:05:17.250 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-03 12:05:17.268 
  command:

    streamlit run /opt/anaconda3/lib/python3.11/site-packages/ipykernel_launcher.py [ARGUMENTS]
2025-07-03 12:05:17.268 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-03 12:05:17.269 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-03 12:05:17.269 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-03 12:05:17.269 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-03 12:05:17.269 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-03 12:05:17.269 Thread 'MainThread': m